# Classic ML Feature Engineering
In this notebook we compare Bag-of-Words and TF-IDF features and inspect how n-grams affect sentiment classification.

**1. Setup and data splits**  
Load cleaned data from the preprocessing artifact and create reproducible train/validation/test splits.

In [6]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import FIGURES_DIR, RESULTS_DIR
from src.data_loader import get_splits
from src.features import get_bow_features, get_tfidf_features

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
metrics_dir = RESULTS_DIR / 'metrics'
metrics_dir.mkdir(parents=True, exist_ok=True)

X_train_text, y_train, X_val_text, y_val, X_test_text, y_test = get_splits(verbose=True)
print('Train size:', X_train_text.shape[0])
print('Val size  :', X_val_text.shape[0])
print('Test size :', X_test_text.shape[0])

Loaded 50,000 rows from preprocessed IMDB dataset.
Split sizes:
  train: 34,999 (70%)
  val  :  7,501 (15%)
  test :  7,500 (15%)
Label distribution (positive class count):
  train: 17,500 / 34,999
  val  :  3,750 / 7,501
  test :  3,750 / 7,500
Train size: 34999
Val size  : 7501
Test size : 7500


**2. BoW vs TF-IDF baseline**  
Use Logistic Regression with the same model settings to compare raw counts vs TF-IDF weighting.

In [7]:
bow = get_bow_features(X_train_text, X_val_text, X_test_text, max_features=10_000, ngram_range=(1, 1))
tfidf = get_tfidf_features(X_train_text, X_val_text, X_test_text, max_features=10_000, ngram_range=(1, 1))

lr_bow = LogisticRegression(max_iter=1000, random_state=42)
lr_bow.fit(bow['train'], y_train)
bow_val_pred = lr_bow.predict(bow['val'])

lr_tfidf = LogisticRegression(max_iter=1000, random_state=42)
lr_tfidf.fit(tfidf['train'], y_train)
tfidf_val_pred = lr_tfidf.predict(tfidf['val'])

baseline_results = [
    {
        'representation': 'BoW (unigram)',
        'val_accuracy': accuracy_score(y_val, bow_val_pred),
        'val_f1': f1_score(y_val, bow_val_pred),
    },
    {
        'representation': 'TF-IDF (unigram)',
        'val_accuracy': accuracy_score(y_val, tfidf_val_pred),
        'val_f1': f1_score(y_val, tfidf_val_pred),
    },
]

baseline_df = pd.DataFrame(baseline_results)
baseline_df

,representation,val_accuracy,val_f1
0,BoW (unigram),0.869484,0.869727
1,TF-IDF (unigram),0.885749,0.886865


**3. Unigram vs bigram (TF-IDF)**  
Evaluate whether adding bigrams improves validation performance by capturing phrase-level cues such as negation.

In [8]:
ngram_results = []

for ngram_range in [(1, 1), (1, 2)]:
    features = get_tfidf_features(
        X_train_text,
        X_val_text,
        X_test_text,
        max_features=10_000,
        ngram_range=ngram_range,
    )

    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(features['train'], y_train)
    val_pred = model.predict(features['val'])

    ngram_results.append(
        {
            'representation': f"ngram_{ngram_range[0]}-{ngram_range[1]}",
            'val_accuracy': accuracy_score(y_val, val_pred),
            'val_f1': f1_score(y_val, val_pred),
        }
    )

ngram_df = pd.DataFrame(ngram_results)
ngram_df

,representation,val_accuracy,val_f1
0,ngram_1-1,0.885749,0.886865
1,ngram_1-2,0.887482,0.888625


**4. Top TF-IDF features (unigram + bigram)**  
Inspect strongest positive and negative coefficients from Logistic Regression to explain model behavior.

In [ ]:
tfidf_bigram = get_tfidf_features(
    X_train_text,
    X_val_text,
    X_test_text,
    max_features=10_000,
    ngram_range=(1, 2),
)

lr_bigram = LogisticRegression(max_iter=1000, random_state=42)
lr_bigram.fit(tfidf_bigram['train'], y_train)

feature_names = tfidf_bigram['vectorizer'].get_feature_names_out()
coefs = lr_bigram.coef_[0]

top_n = 15
top_positive_idx = coefs.argsort()[-top_n:][::-1]
top_negative_idx = coefs.argsort()[:top_n]

top_positive = pd.DataFrame({
    'feature': feature_names[top_positive_idx],
    'weight': coefs[top_positive_idx],
})
top_negative = pd.DataFrame({
    'feature': feature_names[top_negative_idx],
    'weight': coefs[top_negative_idx],
})

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh(top_negative['feature'], top_negative['weight'], color='indianred')
axes[0].set_title('Top negative features')
axes[0].invert_yaxis()

axes[1].barh(top_positive['feature'], top_positive['weight'], color='seagreen')
axes[1].set_title('Top positive features')
axes[1].invert_yaxis()

plt.tight_layout()
feature_plot_path = FIGURES_DIR / 'tfidf_top_features.png'
plt.savefig(feature_plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Saved -> {feature_plot_path}')
top_positive.head(10), top_negative.head(10)

**5. Save metrics for report reuse**

In [ ]:
metrics_df = pd.concat(
    [
        baseline_df.assign(experiment='bow_vs_tfidf'),
        ngram_df.assign(experiment='tfidf_ngram_comparison'),
    ],
    ignore_index=True,
)

metrics_path = metrics_dir / 'checkpoint3_feature_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)
print(f'Saved -> {metrics_path}')
metrics_df

Saved -> C:\projects\imdb-sentiment-analysis\results\metrics\checkpoint3_feature_metrics.csv


,representation,val_accuracy,val_f1,experiment
0,BoW (unigram),0.869484,0.869727,bow_vs_tfidf
1,TF-IDF (unigram),0.885749,0.886865,bow_vs_tfidf
2,"(1, 1)",0.885749,0.886865,tfidf_ngram_comparison
3,"(1, 2)",0.887482,0.888625,tfidf_ngram_comparison
